# Ordered Logistic Regression Results for Adoption Predictors: FAIR² Dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and their `@id` references.

In [ ]:
# Explore available record sets and fields, listing their @id and label.
print("Available Record Sets:")
record_sets = []
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}, label: {field.label if hasattr(field,'label') else field.name}, dataType: {field.data_type}")
    record_sets.append(record_set.id)
    print()
if len(record_sets) == 0:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from each record set into a `DataFrame` for analysis. Reference each record set and field using their `@id`.

In [ ]:
# Load all available record sets into DataFrames
all_dataframes = {}
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        all_dataframes[rs_id] = df
        print(f"Loaded DataFrame for Record Set @id={rs_id} with columns: {df.columns.tolist()[:7]}{'...' if len(df.columns) > 7 else ''}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# For demonstration, select the first record set for EDA if any
selected_record_set = record_sets[0] if record_sets else None
if selected_record_set:
    print(f"\nSample rows from record set @id={selected_record_set}")
    display(all_dataframes[selected_record_set].head())

## 4. Exploratory Data Analysis (EDA)
We will:
- Select a numeric field (by `@id`) and apply filtering, normalization, and grouping.
- Use `@id`-referenced field names dynamically.
- Handle missing values if present.

> Update the below variables to choose which fields to analyze based on the printed record set/field `@id`s above.

In [ ]:
# ---- User: Set these to the actual @id of record set and numeric field ----
record_set_id = selected_record_set  # Use the first record set @id found
df = all_dataframes[record_set_id] if record_set_id else pd.DataFrame()

numeric_field_id = None
group_field_id = None

# Attempt to pick a numeric field by its data type in the metadata
if record_set_id:
    for record_set in dataset.record_sets:
        if record_set.id == record_set_id:
            for field in record_set.fields:
                if (field.data_type or '').lower() in ["float", "number", "integer"] and field.id in df.columns:
                    numeric_field_id = field.id
                    break

# Try to pick a group field (categorical)
if record_set_id and numeric_field_id:
    for record_set in dataset.record_sets:
        if record_set.id == record_set_id:
            for field in record_set.fields:
                if (field.data_type or '').lower() in ["string", "text"] and field.id in df.columns and field.id != numeric_field_id:
                    group_field_id = field.id
                    break
if not numeric_field_id:
    print("No numeric field detected for EDA in the selected record set.")
elif not group_field_id:
    print("No groupable categorical field detected for grouping.")

# Proceed with EDA if possible
if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    # Remove non-numeric/unparseable
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0

    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value):")
    display(filtered_df[[numeric_field_id]].head())

    # Normalization
    if filtered_df.shape[0] > 0:
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean").reset_index()
        print(f"Grouped data by {group_field_id}, showing mean of {numeric_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field could be used for numeric EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn. Here, we plot a histogram of the numeric field and a bar chart if grouping is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field_id and numeric_field_id in df.columns and df[numeric_field_id].notnull().any():
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

if 'grouped_df' in locals():
    plt.figure(figsize=(10, 5))
    sns.barplot(x=group_field_id, y='mean', data=grouped_df)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded the FAIR² dataset with the `mlcroissant` library using its Croissant schema.
- Explored available record sets and fields using their `@id` for precise referencing.
- Extracted records, performed basic cleaning and EDA, and visualized a sample numeric field.

Continue to explore other record sets and fields using the printed field `@id`s, and adjust the EDA steps as needed for your research.